# 🎛️ VGAIN SCAN ANALYSIS


**🔍 Step-by-Step**

**Interactive debugging**



In [ ]:
%load_ext autoreload
%autoreload 2

from waffles.coldboxVD.march_26.analysis.utils import *

In [ ]:
# HD modules analysis 

# 1. Analyzing Vgain scan of 24/03/2026

membrane = 'M3'
channel = 12
led = 1100
bias = 810 #+4,5 OV (FBK)
output_files_comment = ''

vgain = 500
 
input_file = f"/eos/experiment/neutplatform/protodune/experiments/ColdBoxVD/March2026run/HD/BiasVGainScan/afe1bias_{bias:04d}/vgain_{vgain:04d}/led_{led}_ch{channel}/channel_{channel}.dat"

## -- from input file --

with open(f"input/json_files/vgain_parameters_{membrane}_ch{channel}_bias{bias:04d}_led{led}.json", "r") as f:
    dict_params_all = json.load(f)
dict_params_all = {int(k): v for k, v in dict_params_all.items()}

dict_params = dict_params_all[vgain] 

In [ ]:

###Example of dict params with comments

# dict_params = {
#     #Baseline is computed from timetick 0 to baseline_timeticks_limit
#     "baseline_timeticks_limit": 130, 

#     # For the integral computation, you have two possibilities (one must be not null):
#     "deviation_upper_limit": 0.8, # Percentage of the mean wf peak amplitude, to use for the integration (if not used put null)
#     "fixed_integral_limits" : None, # (start, stop) Tuple, containing lower and upper integration limit, if (if not used put null)

#     # Min and max amplitude (in adcs) to use in the hitmap 
#     "heatmap_min": -170,
#     "heatmap_max": 550,

#     # Wavefeform selection for analysis
#     "adcs_threshold": 90, # adcs max (and min) accettable in the baseline range
#     "n_std_baseline": 2, # number of baseline std accetable in the baseline range

#     # Calibration histogram fit 
#     "max_peaks": 7,
#     "prominence": 0.5,
#     "initial_percentage": 0.1,
#     "percentage_step": 0.02,
#     "ch_span_fraction_around_peaks": 0.05,

#     # Additional waveform selection for the integral computation (when "deviation_upper_limit" not None)
#     "time_range_afterpulse_for_meanwf": [200,-1], # Region in which cut for afterpulses, using adcs_threshold
#   }

# input_file = f"/eos/experiment/neutplatform/protodune/experiments/ColdBoxVD/2026March/20260324_sof_ch18to21_vgain_scan_led270_i830_800_930_920_biasmap_remote_r1/vgain_{vgain:04d}/afe2bias_0000/led_{led}_ch{channel}/channel_{channel}.dat" #data analyzied on 25/03 morning 

In [ ]:
validate_analysis_params(dict_params)


## 🔍 **Step-by-Step Analysis**


In [ ]:
wfset_original = create_waveform_set_from_spybuffer(filename=input_file, WFs=40000, length=1024, config_channel=channel)
# plotting_overlap_wf(wfset_original, index_list=[1,2])
# plotting_overlap_wf(wfset_original, n_wf=5)

REMOVING BASELINE

In [ ]:
# BASELINE ANALYSIS + removing 

baseliner_input_parameters = IPDict({
            'baseline_limits': (0,dict_params['baseline_timeticks_limit']),
            'std_cut': 1.,
            'type': 'mean'
        })

checks_kwargs = IPDict({
    'points_no': wfset_original.points_per_wf
})

baseline_analysis_label = 'baseline'

_ = wfset_original.analyse(
    baseline_analysis_label,
    WindowBaseliner,
    baseliner_input_parameters,
    checks_kwargs=checks_kwargs,
    overwrite=True
)

In [ ]:
wfset_original.apply(subtract_baseline, baseline_analysis_label, show_progress=False)
# plotting_overlap_wf(wfset_original, n_wf=2)

In [ ]:
# Dummy analysis for later - it sets the baseline to 0 always

null_baseline_analysis_label = 'null_baseliner'
_ = wfset_original.analyse(
            null_baseline_analysis_label,
            StoreWfAna,
            {'baseline': 0.},
            overwrite=True
        )

STARTING THE FILTERING PROCEDURE

By using the persistance plot, try to find a good sub wfset for the finger plot

In [ ]:
# No filter 

persistance_plot_helper(wfset_original, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)

In [ ]:
# First step - remove waveforms which goes below and up some adcs thresholds in the baseline region (similar to coarse_selection_for_led_calibration from Julio's code)

wfset_1 = WaveformSet.from_filtered_WaveformSet(wfset_original, adcs_threshold_filter, time_range = [0,dict_params['baseline_timeticks_limit']], adcs_minimum_threshold=-dict_params['adcs_threshold'], adcs_maximum_threshold=dict_params['adcs_threshold'])
persistance_plot_helper(wfset_1, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)

In [ ]:
# Second step - look at baseline std distribution to remove noisy waveforms -

average_baseline_std = compute_average_baseline_std(wfset_1, baseline_analysis_label)
#wfset_2 = WaveformSet.from_filtered_WaveformSet(wfset_1, baseline_std_selection, baseline_analysis_label, average_baseline_std, dict_params['n_std_baseline'])
wfset_2 = WaveformSet.from_filtered_WaveformSet(wfset_1, baseline_std_selection, baseline_analysis_label, average_baseline_std, 1)
persistance_plot_helper(wfset_2, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)

In [ ]:
print(f"Original wfset: {len(wfset_original.waveforms)}")
print(f"Adcs cut 1: {len(wfset_1.waveforms)}")
print(f"Std cut: {len(wfset_2.waveforms)}")

wfset_filtered = wfset_2

INTEGRATION WINDOW DEFINITION

Computing the mean waveform 

In [ ]:
# Selecting a subsample of waveforms without afterpulses for the mean waveform computation
wfset_filtered_meanwf = WaveformSet.from_filtered_WaveformSet(wfset_filtered, adcs_threshold_filter, time_range = [dict_params['time_range_afterpulse_for_meanwf'][0],dict_params['time_range_afterpulse_for_meanwf'][1]], adcs_minimum_threshold=-dict_params['adcs_threshold'], adcs_maximum_threshold=dict_params['adcs_threshold'])
persistance_plot_helper(wfset_filtered_meanwf, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)


In [ ]:
mean_wf = wfset_filtered_meanwf.compute_mean_waveform()

plt.figure()
plt.plot(np.array(range(0,1024)), mean_wf.adcs, label="Mean wf")
plt.xlabel("Time ticks")
plt.ylabel("Adcs")
plt.title(f"Mean waveform")
plt.show()

Search for the integration window which maximize the SNR

In [ ]:
# search_integration_window(wfset_filtered, dict_params, channel, deviation_range = (0.5,0.95), deviation_step = 0.05)

Decide which integration window to use

In [ ]:
if dict_params['fixed_integral_limits'] is not None: 
    aux_limits = dict_params['fixed_integral_limits'] #(382, 406)
else: 
    aux_limits = my_integration_window(adcs_array = mean_wf.adcs, deviation_upper_limit=dict_params['deviation_upper_limit'])

In [ ]:
plt.figure()
plt.plot(np.array(range(0,1024)), mean_wf.adcs, label="Mean wf")
plt.axvline(
    aux_limits[0],
    linestyle="--",
    color="red",
    linewidth=1,
    label=f"Lower limit ({aux_limits[0]:.0f})"
)

plt.axvline(
    aux_limits[1],
    linestyle="--",
    color="blue",
    linewidth=1,
    label=f"Upper limit ({aux_limits[1]:.0f})"
)


if dict_params['deviation_upper_limit'] is not None:
    plt.axhline(dict_params['deviation_upper_limit'] * np.max(mean_wf.adcs), 
                linestyle='--', 
                color ='orange', 
                linewidth = 1, 
                label = f"adcs threshold ({dict_params['deviation_upper_limit']*100:.0f}% peak)")

plt.axhline(0, 
            linestyle='--', 
            color ='yellow', 
            linewidth = 1, 
            label = f"Baseline")

plt.legend()
plt.xlabel("Time ticks (AU)")
plt.ylabel("Adcs")
plt.title(f"Mean waveform")
# plt.xlim(350,500)
plt.show()

INTEGRATION ANALYSIS

In [ ]:
integration_analysis_label = 'integration_analysis'

integrator_input_parameters = IPDict({
        'baseline_analysis': null_baseline_analysis_label,
        'inversion': False,
        'int_ll': aux_limits[0],
        'int_ul': aux_limits[1],
        'amp_ll': aux_limits[0],
        'amp_ul': aux_limits[1]
    })

checks_kwargs = IPDict({'points_no': wfset_filtered.points_per_wf})

_ = wfset_filtered.analyse(
    integration_analysis_label,
    WindowIntegrator,
    integrator_input_parameters,
    checks_kwargs=checks_kwargs,
    overwrite=True
)

CALIBRATION HISTOGRAM 

In [ ]:
hist_domain, hist_nbins, hist_bins_width = auto_histogram(wfset_filtered, integration_analysis_label, show_results=False)

In [ ]:
my_grid = coldbox_single_channel_grid(wfset_filtered, config_channel=channel)

my_grid.compute_calib_histos(
            bins_number=hist_nbins, 
            domain=hist_domain, 
            variable='integral',
            analysis_label=integration_analysis_label
        )

fit_peaks_of_ChannelWsGrid(
        my_grid,
        max_peaks=dict_params['max_peaks'], 
        prominence=float(dict_params['prominence']), 
        initial_percentage=dict_params['initial_percentage'],
        percentage_step=dict_params['percentage_step'],
        return_last_addition_if_fail=True,
        fit_type='multigauss_iminuit',
        weigh_fit_by_poisson_sigmas=True,
        ch_span_fraction_around_peaks=dict_params['ch_span_fraction_around_peaks']
    )

# # If you want to play on parameters manually
# fit_peaks_of_ChannelWsGrid(
#     my_grid,
#     max_peaks=7,
#     prominence=0.5,
#     initial_percentage=0.1,
#     percentage_step=0.02,
#     return_last_addition_if_fail=True,
#     fit_type='multigauss_iminuit',
#     weigh_fit_by_poisson_sigmas=True,
#     ch_span_fraction_around_peaks=0.05
# )



fig = plot_ChannelWsGrid(
    my_grid, 
    mode='calibration',
    plot_peaks_fits=True,           
    plot_sum_of_gaussians=True      
)

output_parameters = print_correlated_gaussians_fit_parameters(my_grid, parameters_conversion=True, show=True)

annotation_text = build_calibration_annotation(
            output_parameters,
            hist_domain,
            hist_nbins,
            hist_bins_width
        )

fig.add_annotation(
    text=annotation_text,
    xref = f"x domain" ,
    yref = f"y domain" ,
    x=0.98,
    y=1,
    showarrow=False,
    align="left",
    font=dict(size=15),  
    bgcolor="rgba(255,255,255,0.85)",
    borderwidth=1
)

fig.update_layout(title=f"channel {channel} - vgain {vgain} - led {led}")

fig.show()

In [ ]:
# To count how many wfs we have per peak and see if they are balanced
print(f"Channel {channel} - led {led} - vgain {vgain}")
calibration_histogram_peak_counts(wfset_filtered, output_parameters)

Compute Single Photon electron amplitude

In [ ]:
spe_result = spe_amplitude_computation(wfset_filtered, channel, output_parameters, show_persistance = False, show_spe_hist = True)

Compute dynamic range

In [ ]:
dynamic_range_computation(spe_result['mean'][0])

Compute crosstalk

In [ ]:
# to do....look at federico code # valeria